In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel
from fastapi import HTTPException
from fastapi.responses import JSONResponse
from contextlib import asynccontextmanager
import nest_asyncio
import uvicorn
import threading
import mlflow
from openai import OpenAI, OpenAIError
from loguru import logger

# Apply nest_asyncio to allow nested event loops in Jupyter
nest_asyncio.apply()

/opt/homebrew/Caskroom/miniconda/base/envs/agent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Define the MLflow experiment parameters
PORT = "5001"
EXPERIMENT_NAME = "FastAPI-Ollama"
OLLAMA_API_URL = "http://localhost:11434/v1"
# Define LLM model and parameters
LLM_MODEL = "gpt-oss:20b"
TEMPERATURE = 0.1
MAX_TOKENS = 100

In [ ]:
# MFlow autologging setup
mlflow.openai.autolog()
mlflow.set_tracking_uri(f"http://localhost:{PORT}")
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1774188582234, experiment_id='1', last_update_time=1774188582234, lifecycle_stage='active', name='FastAPI-Ollama', tags={}, workspace='default'>

In [ ]:
def _build_client(base_url: str) -> OpenAI:
    """
    Set up the OpenAI client to connect to the Ollama API.
    """
    if not base_url:
        raise ValueError("OLLAMA_API_URL is not set")
    logger.info(f"Initializing OpenAI client with base_url={base_url!r}")
    return OpenAI(
        base_url=base_url,
        api_key="dummy",  # Ollama doesn't require a real key
    )

client = _build_client(OLLAMA_API_URL)

2026-03-23 15:52:03.186 | INFO     | __main__:_build_client:5 - Initializing AsyncOpenAI client with base_url='http://localhost:11434/v1'


In [ ]:
def llm_generate(prompt: str):
    try:
        logger.info(f"Generating response for prompt: {prompt}")
        response = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        content_response = response.choices[0].message.content
        logger.info(f"Generated response: {content_response}")
        return content_response
    except OpenAIError as e:
        logger.error(f"Error generating response: {e}")
        raise HTTPException(status_code=500, detail="Error generating response from LLM")

In [ ]:
class PromptRequest(BaseModel):
    prompt: str

In [ ]:
app = FastAPI()

@app.get("/")
def read_root():
    return {"Hello": "World"}

@app.post("/generate")
def generate_response(request: PromptRequest):
    try:
        response = llm_generate(request.prompt)
        return {"response": response}
    except Exception as e:
        logger.error(f"Error in /generate endpoint: {e}")
        raise HTTPException(status_code=500, detail="Internal Server Error")
        

In [ ]:
def run():
    uvicorn.run(app, 
                host="0.0.0.0",
                port=8000)

thread = threading.Thread(target=run)
thread.start()

In [ ]:
# To stop the server, you can run the following command in your terminal:
#!kill $(lsof -t -i :8000)